# Simple RAG with RAGAS Evaluation

> **Stack:** LangChain · ChromaDB · OpenAI · RAGAS  
> **Chunking:** Recursive Character Text Splitting  
> **Upload:** `rag_test_document.pdf` (provided)

## Step 1 — Install Dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-openai \
    chromadb pypdf sentence-transformers \
    ragas datasets openai tiktoken

## Step 2 — Set API Key

In [ ]:
import os
from google.colab import userdata

# Store your key in Colab Secrets (key icon on left sidebar) as OPENAI_API_KEY
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Step 3 — Upload the PDF

In [ ]:
from google.colab import files

print("Upload rag_test_document.pdf")
uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {PDF_PATH}")

## Step 4 — Load & Chunk the Document

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Load PDF
loader = PyPDFLoader(PDF_PATH)
raw_docs = loader.load()
print(f"Pages loaded: {len(raw_docs)}")

# Recursive Character Text Splitting
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(raw_docs)
print(f"Chunks created: {len(chunks)}")
print(f"\nSample chunk:\n{chunks[0].page_content[:300]}")

## Step 5 — Build Vector Store (ChromaDB)

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_test_collection",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Vector store ready.")

## Step 6 — Build the RAG Chain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

PROMPT_TEMPLATE = """You are a helpful assistant. Answer the question using ONLY
the context provided below. If the answer is not in the context, say
"I don't know based on the provided document."

Context:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready.")

## Step 7 — Define Test Questions

These questions are grounded in the uploaded PDF so RAGAS can evaluate faithfully.

In [ ]:
test_questions = [
    "By how much have global average temperatures risen above pre-industrial levels as of 2023?",
    "What percentage of global greenhouse gas emissions is carbon dioxide responsible for?",
    "How much has the cost of solar panels dropped over the last decade?",
    "What is the typical efficiency range of a standard residential solar panel?",
    "What capacity factor range do offshore wind turbines typically achieve?",
    "What temperature limit does the Paris Agreement aim to achieve?",
    "What is Sweden's carbon tax per ton of CO2?",
    "What percentage of the EU's total greenhouse gas emissions does the EU ETS cover?",
    "How much has lithium-ion battery cost fallen since 1991?",
    "What is the lifespan of a typical solar panel?",
]

print(f"{len(test_questions)} test questions defined.")

## Step 8 — Generate Answers & Collect Contexts

In [ ]:
answers = []
contexts = []

for q in test_questions:
    # Retrieve relevant docs
    retrieved_docs = retriever.invoke(q)
    ctx = [doc.page_content for doc in retrieved_docs]

    # Generate answer
    answer = rag_chain.invoke(q)

    answers.append(answer)
    contexts.append(ctx)

    print(f"Q: {q}")
    print(f"A: {answer[:120]}...")
    print("-" * 60)

## Step 9 — Prepare RAGAS Dataset

In [ ]:
from datasets import Dataset

# RAGAS expects ground_truth for answer_correctness metric
# These are reference answers aligned with the PDF content
ground_truths = [
    "Global average temperatures have risen by approximately 1.1 degrees Celsius above pre-industrial levels as of 2023.",
    "Carbon dioxide is responsible for approximately 76% of global greenhouse gas emissions.",
    "The cost of solar panels has dropped by more than 89% over the last decade.",
    "A standard residential solar panel has an efficiency of around 15 to 22 percent.",
    "Offshore wind turbines achieve capacity factors of 40 to 60 percent.",
    "The Paris Agreement aims to limit global warming to well below 2 degrees Celsius, with efforts to limit it to 1.5 degrees Celsius.",
    "Sweden has one of the highest carbon taxes in the world at over 130 US dollars per ton of CO2.",
    "The EU ETS covers approximately 40% of the EU's total greenhouse gas emissions.",
    "Lithium-ion battery costs have fallen over 97% since 1991.",
    "Solar panels have a lifespan of 25 to 30 years.",
]

ragas_data = {
    "question":       test_questions,
    "answer":         answers,
    "contexts":       contexts,
    "ground_truth":   ground_truths,
}

dataset = Dataset.from_dict(ragas_data)
print("RAGAS dataset ready:")
print(dataset)

## Step 10 — Run RAGAS Evaluation

| Metric | What it measures |
|---|---|
| **Faithfulness** | Is the answer grounded in the retrieved context? |
| **Answer Relevancy** | Is the answer relevant to the question? |
| **Context Recall** | Does the retrieved context cover the ground truth? |
| **Context Precision** | Is the context free of irrelevant information? |
| **Answer Correctness** | How correct is the answer compared to ground truth? |

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
    answer_correctness,
)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# RAGAS needs its own LLM & embeddings wrappers
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-3.5-turbo", temperature=0))
ragas_emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

metrics = [
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
    answer_correctness,
]

results = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_emb,
    raise_exceptions=False,
)

print("\nRAGAS Evaluation Results:")
print(results)

## Step 11 — Visualise Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Convert to DataFrame
df = results.to_pandas()

# ── Per-question results table ──────────────────────────────────────────────
print("\nPer-question scores:\n")
display(df[["question", "faithfulness", "answer_relevancy",
            "context_recall", "context_precision", "answer_correctness"]])

# ── Aggregate means ──────────────────────────────────────────────────────────
metric_cols = ["faithfulness", "answer_relevancy",
               "context_recall", "context_precision", "answer_correctness"]
means = df[metric_cols].mean()

print("\nAggregate scores (mean):")
print(means.round(4).to_string())

# ── Bar chart ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(means.index, means.values,
              color=["#4C72B0","#DD8452","#55A868","#C44E52","#8172B2"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("RAGAS Evaluation — Aggregate Scores", fontsize=14)
ax.set_xticklabels(means.index, rotation=20, ha="right", fontsize=10)
for bar, val in zip(bars, means.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
            f"{val:.3f}", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.savefig("ragas_scores.png", dpi=150)
plt.show()
print("Chart saved as ragas_scores.png")

## Step 12 — Interactive Query (Optional)

In [ ]:
def ask(question: str) -> str:
    docs = retriever.invoke(question)
    answer = rag_chain.invoke(question)
    print(f"Question : {question}")
    print(f"Answer   : {answer}")
    print(f"\nSources used ({len(docs)}):")
    for i, d in enumerate(docs, 1):
        print(f"  [{i}] Page {d.metadata.get('page', '?')}: {d.page_content[:100]}...")
    return answer

# Try your own questions below
ask("What are the main greenhouse gases?")